# Notebook 16: Rolling evaluatie (voorbeeld)
___
> Dit notebook laat zien hoe elk modelnotebook dezelfde vensters en dezelfde scorefunctie gebruikt. Als voorbeeld-model dient seasonal naive: de laatste gemeten dag wordt 3 keer herhaald
>
> De functies staan in `models/evaluatie.py`. De vensters komen uit `06_Dataprep_v3.ipynb` (stap 4.0), draai dat notebook eerst

### Inladen
___
1. Gedeelde functies inladen

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

#Projectmap zoeken, werkt vanuit elke submap van models/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "prepared_data").exists())
sys.path.append(str(ROOT / "models"))

import evaluatie as ev

2. Reeks en vensters inladen

> `laad_reeks` geeft één reeks als volledig 5-minuten raster. `pow_w` is de echte meting, `pow_imputed` is de meting met de ingevulde train-gaten, `set` zegt of een tijdstip train, val, test of stresstest is

In [2]:
LOCATIE = "huis"
TARGET = "consumption"

df = ev.laad_reeks(LOCATIE, TARGET)

ev.laad_vensters()

,venster,set,start,eind,dagen
0,zomer,val,2025-06-23 00:00:00+02:00,2025-07-03 00:00:00+02:00,10
1,zomer,test,2025-07-03 00:00:00+02:00,2025-07-14 00:00:00+02:00,11
2,herfst,val,2025-10-22 00:00:00+02:00,2025-10-31 00:00:00+01:00,9
3,herfst,test,2025-10-31 00:00:00+01:00,2025-11-10 00:00:00+01:00,10
4,winter,val,2026-01-13 00:00:00+01:00,2026-01-23 00:00:00+01:00,10
5,winter,test,2026-01-23 00:00:00+01:00,2026-02-04 00:00:00+01:00,12
6,lente,val,2026-03-16 00:00:00+01:00,2026-03-26 00:00:00+01:00,10
7,lente,test,2026-03-26 00:00:00+01:00,2026-04-07 00:00:00+02:00,12
8,na_uitval,stresstest,2026-04-25 00:00:00+02:00,2026-05-08 00:00:00+02:00,13


### Trainen
___
1. Trainvoorbeelden kiezen

> `train_origins` geeft de momenten waarop een trainvoorbeeld mag beginnen: 7 dagen input en 3 dagen horizon die volledig in train liggen en geen leeg punt bevatten
>
> Een origin is het rijnummer van het eerste punt dat voorspeld wordt. De input is `df["pow_imputed"][origin - 2016:origin]`, het doel is `df["pow_imputed"][origin:origin + 864]`
>
> Seasonal naive hoeft niet getraind te worden. Bij een echt model train je hier, en daarna blijft het model vast

In [3]:
origins_train = ev.train_origins(df)

print("Aantal trainvoorbeelden:", len(origins_train))

Aantal trainvoorbeelden: 10941


### Rolling evaluatie
___
1. Voorspel-functie

> Het enige wat een modelnotebook zelf moet maken is een functie `voorspel(origins)`. Die krijgt een lijst origins en geeft per origin een voorspelling van 864 stappen (3 dagen) terug, in watt
>
> De functie mag alleen data van vóór de origin gebruiken. In validatie en test is de input niet ingevuld: daar kunnen dus lege punten in zitten. Kan een model daar niet tegen, vul de input dan met `ev.vul_input`, die alleen terugkijkt

In [4]:
voorspel = ev.seasonal_naive(df)

#Zo ziet een eigen functie eruit:
#def voorspel(origins):
#    invoer = np.stack([ev.vul_input(df["pow_w"].to_numpy()[o - ev.INPUT_LEN:o]) for o in origins])
#    return model.predict(invoer)

2. Validatie

> `maak_origins` geeft de vaste voorspelmomenten in de validatievensters, standaard elk uur één (`stap=12`). `rolling_evaluatie` loopt ze op volgorde van tijd af: het model blijft hetzelfde en krijgt steeds iets meer data
>
> Elke voorspelling wordt gescoord op 5, 10, 15, 30 en 60 minuten en op 1, 2 en 3 dagen vooruit. Alleen punten met een echte meting tellen mee

In [5]:
origins_val = ev.maak_origins(df, "val")

detail_val = ev.rolling_evaluatie(voorspel, df, origins_val)

print("Aantal voorspelmomenten:", len(origins_val))
ev.samenvatting(detail_val)

Aantal voorspelmomenten: 653


,aantal,MAE,RMSE
horizon,,,
5 min,582,493.057828,804.580618
10 min,583,450.539604,745.092730
15 min,587,496.041138,824.780953
30 min,562,485.888169,760.909444
60 min,566,481.000247,768.492739
1 dag,577,494.576893,793.935192
2 dagen,583,511.452276,812.046208
3 dagen,587,477.974080,783.191104


3. Per seizoen

> Dezelfde scores, uitgesplitst per venster

In [6]:
ev.samenvatting(detail_val, per_venster=True)["MAE"].unstack("venster").reindex(list(ev.HORIZONS)).round(1)

venster,herfst,lente,winter,zomer
horizon,,,,
5 min,517.9,389.4,551.8,507.0
10 min,421.1,409.3,496.8,457.9
15 min,579.7,359.7,649.0,400.1
30 min,514.0,443.5,482.3,496.8
60 min,467.8,444.5,550.1,452.2
1 dag,492.3,396.0,607.0,465.7
2 dagen,489.8,338.6,690.3,496.1
3 dagen,455.5,350.2,615.2,464.8


4. Test

> Pas draaien als het model en de instellingen vaststaan. De test is alleen voor de eindscore

In [7]:
origins_test = ev.maak_origins(df, "test")

detail_test = ev.rolling_evaluatie(voorspel, df, origins_test)

ev.samenvatting(detail_test)

,aantal,MAE,RMSE
horizon,,,
5 min,703,508.465884,834.640856
10 min,713,494.741585,836.080970
15 min,716,539.135991,895.003276
30 min,710,535.019165,836.855101
60 min,704,515.193919,851.228544
1 dag,707,507.223277,842.643031
2 dagen,711,500.269002,815.977823
3 dagen,713,493.924231,815.420153


5. Opslaan

> De uitkomsten gaan naar `models/results/rolling/`. Het vergelijkingsnotebook leest ze daar allemaal tegelijk in met `ev.laad_resultaten()`

In [8]:
ev.bewaar(detail_val, "seasonal_naive", LOCATIE, TARGET)
ev.bewaar(detail_test, "seasonal_naive", LOCATIE, TARGET)

WindowsPath('c:/Users/Matti/OneDrive/Documenten/Python/Project/Project3_Deel1/models/results/rolling/seasonal_naive_huis_consumption_test.parquet')